In [2]:
"""
This means that instead of opening the website in the browser,
 requests lets the program open it for you.
"""
import requests
from bs4 import BeautifulSoup #Page analysis and data extraction

In [3]:
#response is object
response = requests.get('https://www.britannica.com/technology/information-retrieval')
print(response.status_code)

200


In [4]:
#parse the HTML format
source=response.content
soup = BeautifulSoup(source, 'html.parser')

In [5]:
print(soup.title.text)          # يطبع عنوان الصفحة
print(soup.find('h1').text)     # يطبع أول عنوان رئيسي <h1>
print(soup.find_all('p')[:3])   # يطبع أول 3 فقرات <p>

Information retrieval | Definition, Methods, & Facts | Britannica
information retrieval
[<p>Our editors will review what you’ve submitted and determine whether to revise the article.</p>, <p class="topic-paragraph"><strong><span id="ref1026847"></span>information retrieval</strong>,  recovery of information, especially in a <a class="md-crosslink" data-show-preview="true" href="https://www.britannica.com/technology/database">database</a> stored in a <a class="md-crosslink" data-show-preview="true" href="https://www.britannica.com/technology/computer">computer</a>. Two main approaches are matching words in the query against the database index (keyword searching) and <a class="md-dictionary-link md-dictionary-tt-off mw" data-term="traversing" data-type="MW" href="https://www.merriam-webster.com/dictionary/traversing">traversing</a> the database using <a class="md-crosslink" data-show-preview="true" href="https://www.britannica.com/technology/hypertext">hypertext</a> or hypermedia links. 

In [ ]:
docNum = 0

def formDocument(soup):  # soup --> نتيجة تحليل الصفحة باستخدام BeautifulSoup
    global docNum  # نستخدم المتغير العالمي لتحديث رقم الملف

    with open('Documents\\' + str(docNum+1) + '.txt', 'a', encoding="utf-8") as file:  # a → وضع الإضافة
        docNum += 1
        for p in soup.find_all('p'):  # البحث عن جميع العناصر <p> في الصفحة
            file.write(p.text + '\n\n')  # كتابة نص الفقرة في الملف مع ترك سطر فارغ بين الفقرات
            # print(p.text) 
 


In [7]:
formDocument(soup)

In [ ]:
#pip install validators
#  #التحقق من صحة البيانات قبل استخدامها في النظام أو تخزينها.
"""
Data validation (data verification)
— especially links (URLs), emails, digital addresses (IPs), etc.
"""


'\nData validation (data verification)\n— especially links (URLs), emails, digital addresses (IPs), etc.\n'

In [ ]:
import validators 

homeURLs = []

for a in soup.find_all('a'):  # البحث عن جميع العناصر <a> في الصفحة
    if validators.url(a.get('href')):  # التحقق أن href عبارة عن رابط صالح
        if len(homeURLs) < 5:  # حد أقصى لعدد الصفحات التي سيتم جمعها
            homeURLs.append(a.get('href'))  # إضافة الرابط للقائمة


In [11]:
print("✅ الروابط التي تم استخراجها:")
for link in homeURLs:
    print(link)

✅ الروابط التي تم استخراجها:
https://kids.britannica.com/
http://academic.eb.com
https://www.britannica.com/technology/information-retrieval
https://www.britannica.com/technology/information-retrieval
https://www.britannica.com/technology/information-retrieval/additional-info
https://www.facebook.com/BRITANNICA/
https://x.com/britannica
https://www.britannica.com/technology/information-retrieval
http://nlp.stanford.edu/IR-book/
https://irjaes.com/wp-content/uploads/2020/10/IRJAES-V2N2P214Y17.pdf


In [12]:
for url in homeURLs:
    response = requests.get(url)  # إرسال طلب GET للصفحة
    formDocument(BeautifulSoup(response.content, 'html.parser'))  # تحليل الصفحة وتمريرها للدالة formDocument


In [13]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import os

def formTokens(document, name):
    tokens = set()  # استخدام set لتسريع البحث وضمان عدم التكرار
    words = word_tokenize(document)
    stop_words = set(stopwords.words('english'))  # استدعاء مرة واحدة فقط

    for word in words:
        w = word.lower()
        if w.isalpha() and w not in stop_words:
            tokens.add(w)

    os.makedirs('Tokens', exist_ok=True)  # إنشاء مجلد Tokens إذا لم يكن موجودًا
    with open(os.path.join('Tokens', name), 'w', encoding='utf-8') as file:
        file.write(' '.join(tokens))


In [14]:
import os  # تُستخدم للعمل مع نظام الملفات (مثل قراءة الملفات والمجلدات)

folder_path = 'Documents'  # اسم المجلد اللي فيه الملفات

for filename in os.listdir(folder_path):  # يمر على كل الملفات الموجودة في المجلد
    file_path = os.path.join(folder_path, filename)  # يدمج اسم المجلد مع اسم الملف ليعطي المسار الكامل
    

    with open(file_path, 'r', encoding="utf-8") as file:  # يفتح الملف للقراءة
        document = file.read()  # يقرأ كل محتوى الملف
        formTokens(document, filename)  # يستدعي دالة formTokens لحفظ الكلمات في مجلد Tokens


In [15]:
import os
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer

s_stemmer = SnowballStemmer(language='english')

def formStems(tokensDocument, name):
    # التأكد من وجود المجلد
    os.makedirs('Stems', exist_ok=True)

    tokens = word_tokenize(tokensDocument)  # تقسيم النص إلى كلمات
    stems = set()  # استخدام set لتجنب التكرار

    for word in tokens:
        stems.add(s_stemmer.stem(word))  # إضافة الجذر للقائمة

    # حفظ الجذور في الملف
    with open(os.path.join('Stems', name), 'w', encoding='utf-8') as file:
        file.write(' '.join(stems))


In [16]:
folder_path = 'Tokens'  # المجلد اللي فيه الملفات بعد إزالة التكرار وكلمات التوقف

for filename in os.listdir(folder_path):  # المرور على كل الملفات في المجلد
    file_path = os.path.join(folder_path, filename)  # تكوين المسار الكامل للملف

    with open(file_path, 'r', encoding="utf-8") as f:  # فتح الملف للقراءة
        tokensDocument = f.read()  # قراءة محتوى الملف
        formStems(tokensDocument, filename)  # تحويل الكلمات إلى جذورها وحفظها في مجلد Stems


# IR Pipeline: Data Acquisition, Preprocessing, Indexing, Search, and MAP

This section adds a complete IR pipeline that:

1. Loads .txt documents from C:\\documents
2. Preprocesses (tokenize, stopword removal, stemming, lemmatization)
3. Builds a TF-IDF index
4. Runs queries and ranks by cosine similarity
5. Computes Mean Average Precision (MAP)

Follow the cells below to run the pipeline. Make sure you have .txt files in C:\\documents before running the pipeline.

In [ ]:
# Install and import dependencies (run once)
import sys
import os
import glob
import re

# If packages missing, uncomment and run the line below
# !{sys.executable} -m pip install -q nltk scikit-learn

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

print("Dependencies ready. NLTK corpora downloaded if needed.")

In [ ]:
# Data acquisition: read all .txt files from C:\\documents
import os
from pathlib import Path

DATA_DIR = Path(r"C:\\documents")

corpus = {}  # key: filename, value: content
for filepath in DATA_DIR.glob('*.txt'):
    with open(filepath, 'r', encoding='utf-8') as f:
        corpus[filepath.name] = f.read()

print(f"Loaded {len(corpus)} documents from {DATA_DIR}")

In [ ]:
# Preprocessing: tokenization, stopword removal, stemming, lemmatization
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import string

stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
lm = WordNetLemmatizer()

processed_corpus = {}  # key: filename, value: preprocessed text (whitespace-joined tokens)


def preprocess_text(text):
    # lower
    text = text.lower()
    # remove non-alphabetic characters
    text = re.sub(r"[^a-z\s]", ' ', text)
    # tokenize
    tokens = word_tokenize(text)
    # remove stopwords and punctuation
    tokens = [t for t in tokens if t not in stop_words and t not in string.punctuation]
    # stemming
    stems = [ps.stem(t) for t in tokens]
    # lemmatization (apply on original tokens for better results)
    lemmas = [lm.lemmatize(t) for t in tokens]
    # for representation, we'll use lemmas
    return ' '.join(lemmas)

for fname, content in corpus.items():
    processed_corpus[fname] = preprocess_text(content)

print(f"Preprocessed {len(processed_corpus)} documents")

In [ ]:
# Build TF-IDF matrix
vectorizer = TfidfVectorizer()
docs = [processed_corpus[f] for f in sorted(processed_corpus.keys())]
doc_names = sorted(processed_corpus.keys())

tfidf_matrix = vectorizer.fit_transform(docs)  # shape: (n_docs, n_terms)
print(f"TF-IDF matrix with shape {tfidf_matrix.shape} created")

# For convenience, build a mapping from filename to row index
fname_to_idx = {fname: i for i, fname in enumerate(doc_names)}

In [ ]:
# Query processing and retrieval
queries = ["blockchain implementation", "blockchain types", "blockchain management"]

# preprocess queries using the same preprocess_text function
processed_queries = [preprocess_text(q) for q in queries]

query_vecs = vectorizer.transform(processed_queries)

# compute cosine similarities
sims = cosine_similarity(query_vecs, tfidf_matrix)  # shape: (n_queries, n_docs)

# show top 5 results per query
for qi, q in enumerate(queries):
    ranking = np.argsort(-sims[qi])
    print(f"\nQuery: {q}")
    for rank in ranking[:5]:
        print(f"  - {doc_names[rank]} (score: {sims[qi, rank]:.4f})")


In [ ]:
# Compute Mean Average Precision (MAP)
# For demonstration, the ground truth for each query should be specified.
# Here we will assume the user provides a dictionary of relevant docs per query.

# Example placeholder: you should replace these with the actual relevant files for your dataset
relevance = {
    "blockchain implementation": set(),
    "blockchain types": set(),
    "blockchain management": set()
}


def average_precision(retrieved_indices, relevant_set):
    if not relevant_set:
        return 0.0
    score = 0.0
    hits = 0
    for i, idx in enumerate(retrieved_indices, start=1):
        if doc_names[idx] in relevant_set:
            hits += 1
            score += hits / i
    return score / len(relevant_set)

aps = []
for qi, q in enumerate(queries):
    ranking = list(np.argsort(-sims[qi]))
    ap = average_precision(ranking, relevance[q])
    aps.append(ap)
    print(f"AP for '{q}': {ap:.4f}")

map_score = np.mean(aps)
print(f"\nMAP: {map_score:.4f}")

print("\nNote: Define 'relevance' sets with actual relevant doc filenames to compute meaningful MAP values.")

# Final notes and usage
print("IR pipeline completed in this notebook.\n")
print("Steps to run:\n 1) Ensure .txt files exist in C:\\documents\n 2) Run the cells in order\n 3) Update 'relevance' sets to compute MAP with real ground truth")